# Bottleneck VAE Training - 256 Latent Dimensions
Training BottleneckVAE with BottleneckEncoder and ComplexBottleneckDeconvDecoder using VAE loss with KL divergence

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import time
from tqdm.auto import tqdm
import nibabel as nib

# Runtime reload
import importlib
import models.autoencoder.bottleneck_models
importlib.reload(models.autoencoder.bottleneck_models)

from data.data_ingestion import collect_files, generate_dataframe
from data.dataloader import create_dataloaders
from models.autoencoder.bottleneck_models import (
    BottleneckEncoder,
    ComplexBottleneckDeconvDecoder,
    BottleneckVAE,
)
from models.vae.model import VAELoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Define VAE Loss Function

In [ ]:
class WeightedVAELoss(nn.Module):
    """VAE loss combining weighted reconstruction loss and KL divergence."""
    def __init__(self, weight_mask_path, target_shape=None, beta=0.0005, beta_warmup_steps=5000, free_bits=3.0):
        super().__init__()
        # Load weight mask
        weight_nii = nib.load(weight_mask_path)
        weight_data = weight_nii.get_fdata()
        
        # Convert to tensor
        weight_tensor = torch.from_numpy(weight_data).float()
        
        # Resize if target shape is provided
        if target_shape is not None:
            print(f"Resizing weight mask from {weight_tensor.shape} to {target_shape}")
            weight_tensor = weight_tensor.unsqueeze(0).unsqueeze(0)  # [1, 1, D, H, W]
            weight_tensor = F.interpolate(
                weight_tensor, 
                size=target_shape, 
                mode='trilinear', 
                align_corners=False
            )
            weight_tensor = weight_tensor.squeeze(0).squeeze(0)
        
        # Add batch/channel dimensions
        self.weight_mask = weight_tensor.unsqueeze(0).unsqueeze(0)  # [1, 1, D, H, W]
        
        # Normalize weights to have mean 1 for stable loss scaling
        self.weight_mask = self.weight_mask / self.weight_mask.mean()
        
        self.beta_base = beta
        self.beta_warmup_steps = beta_warmup_steps
        self.current_step = 0
        self.free_bits = free_bits
        
        print(f"Weight mask shape: {self.weight_mask.shape}")
        print(f"Weight range: [{self.weight_mask.min():.4f}, {self.weight_mask.max():.4f}]")
    
    def forward(self, recon_x, x, mu, log_var):
        # Move mask to same device as predictions
        if self.weight_mask.device != recon_x.device:
            self.weight_mask = self.weight_mask.to(recon_x.device)
        
        # Resize mask if needed to match prediction shape
        if self.weight_mask.shape[2:] != recon_x.shape[2:]:
            resized_mask = F.interpolate(
                self.weight_mask,
                size=recon_x.shape[2:],
                mode='trilinear',
                align_corners=False
            )
        else:
            resized_mask = self.weight_mask
        
        # Weighted reconstruction loss
        squared_error = (recon_x - x) ** 2
        weighted_error = squared_error * resized_mask
        recon_loss = weighted_error.mean()
        
        # KL divergence with free bits
        kl_raw = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=1)
        kl_free = torch.maximum(kl_raw - self.free_bits, torch.zeros_like(kl_raw))
        kl_loss = torch.mean(kl_free)
        
        # Calculate beta with cyclical warmup
        if self.beta_warmup_steps > 0:
            cycle_length = self.beta_warmup_steps // 2
            cycle_position = (self.current_step % cycle_length) / cycle_length
            if self.current_step // cycle_length % 2 == 0:  # Even cycles: warmup
                beta = self.beta_base * cycle_position
            else:  # Odd cycles: constant
                beta = self.beta_base
        else:
            beta = self.beta_base
        
        # Increment step counter
        self.current_step += 1
        
        # Total loss
        total_loss = recon_loss + beta * kl_loss
        
        return total_loss, recon_loss, kl_loss, beta

# Create weighted VAE loss criterion
criterion = WeightedVAELoss("data/masks/weightMatrix.nii", target_shape=(64, 128, 128))
print("Weighted VAE loss criterion created")

## Data Setup

In [ ]:
root_dir = Path(".")
images_dir = root_dir / "data" / "Images"
mask_path = root_dir / "data" / "masks" / "rmask_ICV.nii"

# Configuration
data_dir = "data/Images"
mask_path = "data/masks/rmask_ICV.nii"
batch_size = 4
output_dir = "output/Experiments/BottleneckVAETraining"

# Create output directory
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Prepare data
print("Collecting files...")
included_files, excluded_files = collect_files(data_dir)
print(f"Found {len(included_files)} valid files, excluded {len(excluded_files)} files")

# Generate dataframe
print("Generating dataframe...")
df = generate_dataframe(included_files)
display(df.head())

# Create dataloaders
print("Creating dataloaders...")
train_loader, val_loader = create_dataloaders(
    df, batch_size=batch_size, train_split=0.8, 
    on_demand=True, mask_path=mask_path,
    num_workers=2
)

print(f"Data preparation complete. Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}")

## Model Setup

In [ ]:
target_shape = (64, 128, 128)
latent_dim = 256

def build_vae_model(latent_dim=256):
    """Build BottleneckVAE model"""
    model = BottleneckVAE(
        BottleneckEncoder(initial_filters=4, latent_dim=latent_dim, bottleneck_shape=(1,1,1)),
        ComplexBottleneckDeconvDecoder(latent_dim=latent_dim, target_shape=target_shape),
        latent_dim=latent_dim
    )
    return model.to(device)

model = build_vae_model(latent_dim=latent_dim)
num_params = count_trainable(model)
print(f"Model Parameters: {num_params / 1e6:.2f}M")

## Training Loop

In [ ]:
def train_vae_epoch(model, train_loader, optimizer, criterion, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    pbar = tqdm(train_loader, desc="Training", leave=False)
    for batch_idx, (data, _) in enumerate(pbar):
        data = data.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        recon, mu, log_var = model(data)
        
        # Compute loss
        loss, recon_loss, kl_loss, beta = criterion(recon, data, mu, log_var)
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
        
        pbar.set_postfix({
            'loss': f'{loss.item():.6f}',
            'recon': f'{recon_loss.item():.6f}',
            'kl': f'{kl_loss.item():.6f}',
            'beta': f'{beta:.6f}'
        })
    
    avg_loss = total_loss / len(train_loader)
    avg_recon = total_recon / len(train_loader)
    avg_kl = total_kl / len(train_loader)
    
    return avg_loss, avg_recon, avg_kl

def validate_vae(model, val_loader, criterion, device):
    """Validate model"""
    model.eval()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="Validation", leave=False)
        for data, _ in pbar:
            data = data.to(device)
            
            # Forward pass
            recon, mu, log_var = model(data)
            
            # Compute loss
            loss, recon_loss, kl_loss, beta = criterion(recon, data, mu, log_var)
            
            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()
            
            pbar.set_postfix({
                'loss': f'{loss.item():.6f}',
                'recon': f'{recon_loss.item():.6f}',
                'kl': f'{kl_loss.item():.6f}'
            })
    
    avg_loss = total_loss / len(val_loader)
    avg_recon = total_recon / len(val_loader)
    avg_kl = total_kl / len(val_loader)
    
    return avg_loss, avg_recon, avg_kl

In [ ]:
# Training configuration
num_epochs = 150
learning_rate = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10, verbose=True
)

# Training history
train_losses = []
val_losses = []
train_recon_losses = []
train_kl_losses = []
val_recon_losses = []
val_kl_losses = []

best_val_loss = float('inf')
patience_counter = 0
early_stopping_patience = 20

model_name = f"bottleneck_vae_lat{latent_dim}"
checkpoint_dir = os.path.join(output_dir, model_name)
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"Starting training for {num_epochs} epochs...")
start_time = time.time()

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    
    # Train
    train_loss, train_recon, train_kl = train_vae_epoch(model, train_loader, optimizer, criterion, device)
    train_losses.append(train_loss)
    train_recon_losses.append(train_recon)
    train_kl_losses.append(train_kl)
    
    # Validate
    val_loss, val_recon, val_kl = validate_vae(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    val_recon_losses.append(val_recon)
    val_kl_losses.append(val_kl)
    
    print(f"Train Loss: {train_loss:.6f} (Recon: {train_recon:.6f}, KL: {train_kl:.6f})")
    print(f"Val Loss: {val_loss:.6f} (Recon: {val_recon:.6f}, KL: {val_kl:.6f})")
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Early stopping and checkpointing
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        best_model_path = os.path.join(checkpoint_dir, f"{model_name}_best.pth")
        torch.save(model.state_dict(), best_model_path)
        print(f"✓ Best model saved: {best_model_path}")
    else:
        patience_counter += 1
    
    # Save periodic checkpoint
    if (epoch + 1) % 10 == 0:
        checkpoint_path = os.path.join(checkpoint_dir, f"{model_name}_epoch{epoch+1}.pth")
        torch.save(model.state_dict(), checkpoint_path)
        print(f"✓ Checkpoint saved: {checkpoint_path}")
    
    # Early stopping
    if patience_counter >= early_stopping_patience:
        print(f"\nEarly stopping triggered after {epoch+1} epochs")
        break

elapsed_time = time.time() - start_time
print(f"\n✓ Training completed in {elapsed_time:.1f}s")
print(f"Best validation loss: {best_val_loss:.6f}")

## Results Summary

In [ ]:
# Create results dataframe
results_df = pd.DataFrame({
    'Epoch': range(1, len(train_losses) + 1),
    'Train Loss': train_losses,
    'Val Loss': val_losses,
    'Train Recon': train_recon_losses,
    'Train KL': train_kl_losses,
    'Val Recon': val_recon_losses,
    'Val KL': val_kl_losses,
})

# Save results
results_path = os.path.join(checkpoint_dir, f"{model_name}_results.csv")
results_df.to_csv(results_path, index=False)
print(f"Results saved to: {results_path}")

display(results_df.tail(10))

## Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Total loss
axes[0, 0].plot(train_losses, label='Train', linewidth=2)
axes[0, 0].plot(val_losses, label='Val', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Total Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Reconstruction loss
axes[0, 1].plot(train_recon_losses, label='Train', linewidth=2)
axes[0, 1].plot(val_recon_losses, label='Val', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Reconstruction Loss')
axes[0, 1].set_title('Reconstruction Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# KL loss
axes[1, 0].plot(train_kl_losses, label='Train', linewidth=2)
axes[1, 0].plot(val_kl_losses, label='Val', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('KL Loss')
axes[1, 0].set_title('KL Divergence Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Loss ratio
recon_kl_ratio = [r / (k + 1e-8) for r, k in zip(val_recon_losses, val_kl_losses)]
axes[1, 1].plot(recon_kl_ratio, linewidth=2, color='purple')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recon / KL Ratio')
axes[1, 1].set_title('Reconstruction to KL Loss Ratio')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(checkpoint_dir, f"{model_name}_training_curves.png")
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"Plot saved to: {plot_path}")
plt.show()

## Load Best Model and Test

In [ ]:
# Load best model
best_model = build_vae_model(latent_dim=latent_dim)
best_model_path = os.path.join(checkpoint_dir, f"{model_name}_best.pth")
best_model.load_state_dict(torch.load(best_model_path))
print(f"Loaded best model from: {best_model_path}")

# Test on a few samples
best_model.eval()
with torch.no_grad():
    # Get a batch from validation set
    test_data, _ = next(iter(val_loader))
    test_data = test_data.to(device)
    
    # Forward pass
    recon, mu, log_var = best_model(test_data)
    
    # Compute loss
    loss, recon_loss, kl_loss, beta = criterion(recon, test_data, mu, log_var)
    
    print(f"\nTest Batch Results:")
    print(f"Total Loss: {loss.item():.6f}")
    print(f"Reconstruction Loss: {recon_loss.item():.6f}")
    print(f"KL Loss: {kl_loss.item():.6f}")
    print(f"Beta: {beta:.6f}")
    print(f"Latent Mean: {mu.mean().item():.6f}")
    print(f"Latent Std: {mu.std().item():.6f}")